In [57]:
# !pip install rapidfuzz

In [36]:
from rapidfuzz import fuzz
import re
import json
from pprint import pprint

In [75]:
def sliding_window_score(target, candidate, window_factor=1.5, step=None, scorer=fuzz.token_sort_ratio): #есть идеи как улучшить, смотрим на результат
    '''
    target: короткая строка
    candidate: длинная строка
    window_factor: размер окна = len(target) * window_factor
    step: шаг окна (по умолчанию = max(1, len(target)//4))
    '''
    
    def normalize(s):
        '''базовая нормализация'''
        s = s.lower().replace('ё', 'е')
        s = re.sub(r'[^\w\s\.,-:;]', ' ', s)   # всё, что не буква/цифра/пробел → пробел
        s = re.sub(r'(?<!\d)([.,;:])(?!\d)[ \t]*', r' \1 ', s)
        s = re.sub(r'\s+', ' ', s).strip()
        return s      

    target, candidate = normalize(target), normalize(candidate)

    best = 0.0
    
    n, m = len(target), len(candidate)
    if m <= n * window_factor:
        best = scorer(target, candidate)  # кандидат не длиннее окна — сравниваем целиком

    w = int(n * window_factor)
    if step is None:
        step = max(1, n // 4)

    if not best:
        for start in range(0, m - w + 1, step):
            chunk = candidate[start:start + w]
            best = max(best, scorer(target, chunk))
    return best
    
# target = 'нечёткий поиск;'
# long_candidate = 'Это очень длинный текст, в котором где-то в середине упоминается нечёткий поиск; и потом много других слов.'

# print(sliding_window_score(target, long_candidate, step=1))          # окно 1.5x
# print(sliding_window_score(target, long_candidate, 2, step=1))     # окно 2x
# print(sliding_window_score(target, long_candidate, 1.2, step=1))     # окно 1.2x — точнее, но медленнее

print(sliding_window_score('Unit 1 . 3 . 2', 'Unit 1 . 3 . 2', step=1))
print(sliding_window_score('x y', 'xex', step=1))

100.0
33.333333333333336


In [76]:
# открыть clean.json
with open(r'../gold/clean.json', encoding='utf-8') as f:
    y_all = json.load(f)

# открыть один gold X
with open('../gold/parsed_en/ED.A023.MII.&.&&&.02026-II (eng).json', encoding='utf-8') as f:
    x = json.load(f)

y = y_all['ED.A023.MII.&.&&&.02026-II']

In [79]:
pprint(y)

{'block': 'Unit 1.3.2',
 'commission': ['Kochurov Dmitrii', 'Voronchikhina Iana', 'Sukhomlinova Alla'],
 'conclusion': 'Rejected',
 'date': '2026-07-27',
 'documents': ['Quality certificate/Сертификат качества No 4287738 dated '
               '04.2026 (1 page)',
               'Quality certificate/Сертификат качества No 4290047 dated '
               '04.2026 (1 page)',
               'Quality certificate/Сертификат качества No 4347511 dated '
               '04.2026 (1 page)',
               'Quality certificate/Сертификат качества No 4371877 dated '
               '07.2026 (1 page)',
               'Quality certificate/Сертификат качества No 4169148 dated '
               '01.2026 (1 page)',
               'Quality certificate/Сертификат качества No 4169149 dated '
               '01.2026 (1 page)',
               'Quality certificate/Сертификат качества No 4180674 dated '
               '01.2026 (1 page)',
               'Quality certificate/Сертификат качества No 4370324 dated '
 

In [17]:
ord('`')

96

In [53]:
from rapidfuzz import fuzz, distance

target = 'нечёткий поиск;'
candidates = ['нечеткий понятно поиск нечеткий понятно поиск', 'поиск нечеткий;_______________________', 'поиск нечеткий; _______________________', 'нечеткийпоиск', 'поискнечеткий', 'чёткий поиск', 'поиск по тексту', 'нечёткий пойск']

for c in candidates:
    # нормализованное сходство 0..100
    # score = fuzz.ratio(target, c)
    # score = fuzz.WRatio(target, c)
    # score = fuzz.partial_token_set_ratio(target, c)
    score = sliding_window_score(target, c)
    print(f'{score:5.1f}  {c}')

 76.9  нечеткий понятно поиск нечеткий понятно поиск
 66.7  поиск нечеткий;_______________________
 66.7  поиск нечеткий; _______________________
 84.6  нечеткийпоиск
 66.7  поискнечеткий
100.0  чёткий поиск
 50.0  поиск по тексту
 92.9  нечёткий пойск


In [41]:
dir(fuzz)

['AVX2',
 'QRatio',
 'SSE2',
 'WRatio',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 '_impl',
 '_imported',
 'annotations',
 'contextlib',
 'os',
 'partial_ratio',
 'partial_ratio_alignment',
 'partial_token_ratio',
 'partial_token_set_ratio',
 'partial_token_sort_ratio',
 'ratio',
 'supports',
 'token_ratio',
 'token_set_ratio',
 'token_sort_ratio']

In [49]:
from rapidfuzz import fuzz

def sliding_window_score(target, candidate, window_factor=1.5, step=None, scorer=fuzz.token_sort_ratio):
    '''
    target: короткая строка
    candidate: длинная строка
    window_factor: размер окна = len(target) * window_factor
    step: шаг окна (по умолчанию = max(1, len(target)//4))
    '''
    n, m = len(target), len(candidate)
    if m <= n * window_factor:
        return scorer(target, candidate)  # кандидат не длиннее окна — сравниваем целиком

    w = int(n * window_factor)
    if step is None:
        step = max(1, n // 4)

    best = 0.0
    for start in range(0, m - w + 1, step):
        chunk = candidate[start:start + w]
        best = max(best, scorer(target, chunk))
    return best
    
target = 'нечёткий поиск;'
long_candidate = 'Это очень длинный текст, в котором где-то в середине упоминается нечёткий поиск; и потом много других слов.'

print(sliding_window_score(target, long_candidate))          # окно 1.5x
print(sliding_window_score(target, long_candidate, 2))     # окно 2x
print(sliding_window_score(target, long_candidate, 1.2))     # окно 1.2x — точнее, но медленнее

81.08108108108108
68.18181818181819
93.75


In [5]:
from rapidfuzz import fuzz

def sliding_window_score(target, candidate, window_factor=1.5, step=None, scorer=fuzz.token_sort_ratio):
    '''
    target: короткая строка
    candidate: длинная строка
    window_factor: размер окна = len(target) * window_factor
    step: шаг окна (по умолчанию = max(1, len(target)//4))
    '''
    n, m = len(target), len(candidate)
    print('m', '---', m)
    if m <= n * window_factor:
        return scorer(target, candidate)  # кандидат не длиннее окна — сравниваем целиком

    w = int(n * window_factor)
    print('w', '---', w)
    if step is None:
        step = max(1, n // 4)
        print('step', '---', step)

    best = 0.0
    for start in range(0, m - w + 1, step):
        print('start', '---', start, '   ', 'finish', '---', start + w)
        
        chunk = candidate[start:start + w]
        best = max(best, scorer(target, chunk))
    return best
    
target = 'нечёткий поиск;'
long_candidate = 'Это очень длинный текст, в котором где-то в середине упоминается нечёткий поиск; и потом много других слов.'

print(sliding_window_score(target, long_candidate, step=1))          # окно 1.5x
# print(sliding_window_score(target, long_candidate, 2))     # окно 2x
# print(sliding_window_score(target, long_candidate, 1.2))     # окно 1.2x — точнее, но медленнее

m --- 107
w --- 22
start --- 0     finish --- 22
start --- 1     finish --- 23
start --- 2     finish --- 24
start --- 3     finish --- 25
start --- 4     finish --- 26
start --- 5     finish --- 27
start --- 6     finish --- 28
start --- 7     finish --- 29
start --- 8     finish --- 30
start --- 9     finish --- 31
start --- 10     finish --- 32
start --- 11     finish --- 33
start --- 12     finish --- 34
start --- 13     finish --- 35
start --- 14     finish --- 36
start --- 15     finish --- 37
start --- 16     finish --- 38
start --- 17     finish --- 39
start --- 18     finish --- 40
start --- 19     finish --- 41
start --- 20     finish --- 42
start --- 21     finish --- 43
start --- 22     finish --- 44
start --- 23     finish --- 45
start --- 24     finish --- 46
start --- 25     finish --- 47
start --- 26     finish --- 48
start --- 27     finish --- 49
start --- 28     finish --- 50
start --- 29     finish --- 51
start --- 30     finish --- 52
start --- 31     finish --- 53

In [16]:
import re

def normalize(s):
    s = s.lower().replace('ё', 'е')
    s = re.sub(r'[^\w\s\.,]', ' ', s)   # всё, что не буква/цифра/пробел → пробел
    s = re.sub(r'(?<!\d)([.,])(?!\d)[ \t]*', r' \1 ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

normalize('safsed парк 123.56 sd,fdsf')

'safsed парк 123.56 sd , fdsf'